# EcoSort: Paper 3's segmentation pipeline (Kumar et al., Sci Rep 2026) + our models

Paper 3 = hybrid segmentation (global + adaptive + Otsu thresholds + K-means k=2, majority vote, watershed boundaries removed, mask multiplied onto the image) followed by a CNN. This notebook:
1. implements their Algorithm 1 with OpenCV,
2. trains ResNet101 on the segmented images (their ablation says about 94.7%),
3. trains our ConvNeXt-Tiny on segmented images and checks whether segmented-input models improve our ensemble on **validation**.

Colab, Runtime > T4 GPU. Run all cells top to bottom (about 1 to 1.5 hours). Test is not touched here.
The paper does not state the threshold parameters (T, window, C), so the values below are my choices; treat results as an approximate replication.

In [ ]:
# ===== RESTORE CELL (no training) =====
import subprocess, sys, os, glob, shutil, json, math, time, random, io, contextlib, gc
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "scikit-learn"], check=False)
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, recall_score

assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then re-run."
device = "cuda"; torch.backends.cudnn.benchmark = True
SEED, IMG_SIZE, BATCH_SIZE = 42, 224, 32
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
WORK, DATA_DIR = "/content/work", "/content/realwaste/RealWaste"
os.makedirs(WORK, exist_ok=True)

# 1) dataset
if not os.path.isdir("/content/realwaste"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sam-single/realwaste.git", "/content/realwaste"], check=True)

# 2) restore saved files from Drive
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
DST = "/content/drive/MyDrive/EcoSort_ml"
for fn in os.listdir(DST):
    if os.path.isfile(f"{DST}/{fn}") and not os.path.exists(f"{WORK}/{fn}"): shutil.copy2(f"{DST}/{fn}", f"{WORK}/{fn}")
print("Restored files:", len(os.listdir(WORK)))

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

# 3) splits + image cache
train_df, val_df, test_df = [pd.read_csv(f"{WORK}/{n}.csv") for n in ("train", "val", "test")]
class_to_idx = json.load(open(f"{WORK}/class_to_idx.json"))
classes = sorted(class_to_idx, key=class_to_idx.get)

def load_resize(rel_path):
    im = Image.open(os.path.join(DATA_DIR, rel_path)).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
    return np.asarray(im, dtype=np.uint8)
def build_cache(df, name):
    cf = f"{WORK}/cache_{name}_{IMG_SIZE}.npy"
    if os.path.exists(cf): return np.load(cf)
    with ThreadPoolExecutor(8) as ex: arr = np.stack(list(ex.map(load_resize, df["path"].tolist())))
    np.save(cf, arr); return arr
X_train, X_val, X_test = build_cache(train_df, "train"), build_cache(val_df, "val"), build_cache(test_df, "test")

# 4) GPU loader (same as Cell 5)
class GPULoader:
    def __init__(self, X, y, batch_size, train, seed=SEED):
        self.X = torch.from_numpy(X).permute(0, 3, 1, 2).contiguous().to(device)
        self.y = torch.from_numpy(np.asarray(y)).long().to(device)
        self.bs, self.train = batch_size, train
        self.mean = torch.tensor(MEAN, device=device).view(1, 3, 1, 1); self.std = torch.tensor(STD, device=device).view(1, 3, 1, 1)
        self.g = torch.Generator(device=device); self.g.manual_seed(seed)
        self.cos_t = torch.tensor([1., 0., -1., 0.], device=device); self.sin_t = torch.tensor([0., 1., 0., -1.], device=device)
    def __len__(self):
        n = len(self.y); return n // self.bs if self.train else math.ceil(n / self.bs)
    def _u(self, n, lo=0.0, hi=1.0): return lo + (hi - lo) * torch.rand(n, device=device, generator=self.g)
    def _augment(self, x):
        B = x.shape[0]
        k = torch.randint(0, 4, (B,), device=device, generator=self.g); cos, sin = self.cos_t[k], self.sin_t[k]
        s = self._u(B, 0.7, 1.0).sqrt(); ar = self._u(B, 0.9, 1.1).sqrt()
        sx, sy = (s * ar).clamp(max=1.0), (s / ar).clamp(max=1.0)
        fx = (self._u(B) < 0.5).float() * 2 - 1; fy = (self._u(B) < 0.5).float() * 2 - 1
        a, b = fx * sx, fy * sy
        ex = cos.abs() * sx + sin.abs() * sy; ey = sin.abs() * sx + cos.abs() * sy
        tx = self._u(B, -1, 1) * (1 - ex); ty = self._u(B, -1, 1) * (1 - ey)
        theta = torch.stack([torch.stack([cos * a, -sin * b, tx], 1), torch.stack([sin * a, cos * b, ty], 1)], 1)
        grid = F.affine_grid(theta, list(x.shape), align_corners=False)
        x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
        x = x * self._u(B, 0.8, 1.2).view(B, 1, 1, 1)
        gray = (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]); m = gray.mean(dim=(1, 2, 3), keepdim=True)
        x = (x - m) * self._u(B, 0.8, 1.2).view(B, 1, 1, 1) + m
        gray = (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3])
        x = gray + (x - gray) * self._u(B, 0.9, 1.1).view(B, 1, 1, 1)
        return x.clamp(0, 1)
    def __iter__(self):
        n = len(self.y)
        idx = torch.randperm(n, device=device, generator=self.g) if self.train else torch.arange(n, device=device)
        for i in range(len(self)):
            b = idx[i * self.bs:(i + 1) * self.bs]; x = self.X[b].float().div_(255)
            if self.train: x = self._augment(x)
            yield ((x - self.mean) / self.std).contiguous(memory_format=torch.channels_last), self.y[b]

train_loader = GPULoader(X_train, train_df["y"].values, BATCH_SIZE, True)
val_loader   = GPULoader(X_val,   val_df["y"].values,   64, False)
test_loader  = GPULoader(X_test,  test_df["y"].values,  64, False)

# 5) engine (same as Cell 4)
val_criterion = nn.CrossEntropyLoss()
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); logits_all, y_all = [], []; tot = 0.0
    for xb, yb in loader:
        with torch.autocast("cuda", dtype=torch.float16): out = model(xb)
        tot += val_criterion(out.float(), yb).item() * len(yb); logits_all.append(out.float().cpu()); y_all.append(yb.cpu())
    logits = torch.cat(logits_all); y = torch.cat(y_all).numpy(); pred = logits.argmax(1).numpy()
    return {"loss": tot/len(y), "acc": accuracy_score(y, pred), "f1": f1_score(y, pred, average="macro"), "logits": logits, "y": y, "pred": pred}

def run_experiment(arch, tag, head_epochs=3, ft_epochs=25, patience=7, lr_head=1e-3, lr_backbone=1e-4,
                   weight_decay=0.01, label_smoothing=0.0, use_class_weights=False, drop_rate=0.2):
    seed_everything(SEED)
    model = timm.create_model(arch, pretrained=True, num_classes=len(classes), drop_rate=drop_rate)
    model = model.to(device).to(memory_format=torch.channels_last)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"[{tag}] {arch} | {n_params:.1f}M params | class_weights={use_class_weights} | label_smoothing={label_smoothing}")
    cw = torch.tensor(np.load(f"{WORK}/class_weights.npy"), dtype=torch.float32, device=device) if use_class_weights else None
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=label_smoothing)
    scaler = torch.amp.GradScaler("cuda")
    head_params = list(model.get_classifier().parameters()); head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    def train_epoch(opt, sched=None):
        model.train(); tl, n, t0 = 0.0, 0, time.time()
        for xb, yb in train_loader:
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16): out = model(xb)
            loss = criterion(out.float(), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sched: sched.step()
            tl += loss.item() * len(yb); n += len(yb)
        return tl / n, n / (time.time() - t0)
    history, best_f1, best_ep, bad = [], -1.0, 0, 0
    ckpt = f"{WORK}/ckpt_{tag}.pt"
    def log(ep, stage, trl, ips, v):
        history.append({"epoch": ep, "stage": stage, "train_loss": trl, "val_loss": v["loss"], "val_acc": v["acc"], "val_f1": v["f1"]})
        print(f"ep {ep:02d} [{stage:4s}] train_loss {trl:.3f} | val_loss {v['loss']:.3f} acc {v['acc']:.4f} f1 {v['f1']:.4f} | {ips:.0f} img/s")
    for p in backbone_params: p.requires_grad = False
    opt = torch.optim.AdamW(head_params, lr=lr_head, weight_decay=weight_decay)
    for ep in range(1, head_epochs + 1):
        trl, ips = train_epoch(opt); v = evaluate(model, val_loader); log(ep, "head", trl, ips, v)
    for p in backbone_params: p.requires_grad = True
    opt = torch.optim.AdamW([{"params": backbone_params, "lr": lr_backbone}, {"params": head_params, "lr": lr_head}], weight_decay=weight_decay)
    total, warm = ft_epochs * len(train_loader), len(train_loader)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.01, 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm)))))
    for ep in range(head_epochs + 1, head_epochs + ft_epochs + 1):
        trl, ips = train_epoch(opt, sched); v = evaluate(model, val_loader); log(ep, "full", trl, ips, v)
        if v["f1"] > best_f1: best_f1, best_ep, bad = v["f1"], ep, 0; torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= patience: print(f"Early stop at epoch {ep} (best epoch {best_ep}, val F1 {best_f1:.4f})"); break
    model.load_state_dict(torch.load(ckpt)); v = evaluate(model, val_loader)
    print(f"\nBEST epoch {best_ep} | val acc {v['acc']:.4f} | val macro-F1 {v['f1']:.4f} | val loss {v['loss']:.3f}")
    print(classification_report(v["y"], v["pred"], target_names=classes, digits=3))
    res = {"tag": tag, "arch": arch, "params_M": n_params, "best_epoch": best_ep, "val_acc": v["acc"], "val_f1": v["f1"],
           "img_size": IMG_SIZE, "class_weights": use_class_weights, "label_smoothing": label_smoothing, "history": history}
    json.dump(res, open(f"{WORK}/results_{tag}.json", "w"))
    return model, res

@torch.no_grad()
def infer_speed(m, loader, reps=3):
    m.eval(); torch.cuda.synchronize(); t0 = time.time(); n = 0
    for _ in range(reps):
        for xb, yb in loader:
            with torch.autocast("cuda", dtype=torch.float16): m(xb)
            n += len(yb)
    torch.cuda.synchronize(); return n / (time.time() - t0)

def sweep(models, suffix="224", **kw):
    for arch in models:
        tag = f"{arch}_{suffix}"
        if os.path.exists(f"{WORK}/results_{tag}.json"): print(f"[skip] {tag} already done"); continue
        train_loader.g.manual_seed(SEED)
        t0 = time.time(); buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf): m, r = run_experiment(arch, tag, **kw)
        except Exception:
            print("FAILED:", tag); print(buf.getvalue()[-1500:]); raise
        mins = (time.time() - t0) / 60
        v = evaluate(m, val_loader); rec = recall_score(v["y"], v["pred"], average=None)
        r.update({"train_minutes": mins, "infer_img_s": infer_speed(m, val_loader), "val_recall": dict(zip(classes, rec.tolist()))})
        json.dump(r, open(f"{WORK}/results_{tag}.json", "w"))
        print(f"{tag:26s} params {r['params_M']:5.1f}M | best ep {r['best_epoch']:2d}/{r['history'][-1]['epoch']} | val acc {r['val_acc']:.4f} F1 {r['val_f1']:.4f} | {mins:.1f} min | infer {r['infer_img_s']:.0f} img/s")
        print("   recall:", {k[:8]: round(x, 2) for k, x in r["val_recall"].items()})
        del m; torch.cuda.empty_cache()

def show_table():
    rows = []
    for f in sorted(glob.glob(f"{WORK}/results_*.json")):
        r = json.load(open(f))
        rows.append({"run": r["tag"], "params_M": round(r.get("params_M", float("nan")), 1), "best_ep": r.get("best_epoch", float("nan")), "val_acc": round(r["val_acc"], 4),
                     "val_f1": round(r["val_f1"], 4), "min": round(r.get("train_minutes", float("nan")), 1), "infer_img_s": round(r.get("infer_img_s", float("nan")), 0)})
    print(pd.DataFrame(rows).sort_values("val_f1", ascending=False).to_string(index=False))

# 6) verification
print(f"\nSplits: train {len(train_df)} | val {len(val_df)} | test {len(test_df)} (expected 3327 / 712 / 713)")
print(f"Checkpoints restored: {len(glob.glob(WORK + '/ckpt_*.pt'))} (expected 13)\n")
show_table()

In [ ]:

import copy
def cache_for(size):
    out = []
    for df, name in ((train_df, "train"), (val_df, "val"), (test_df, "test")):
        cf = f"{WORK}/cache_{name}_{size}.npy"
        if os.path.exists(cf):
            out.append(np.load(cf)); continue
        def _load(p, size=size):
            im = Image.open(os.path.join(DATA_DIR, p)).convert("RGB").resize((size, size), Image.BICUBIC)
            return np.asarray(im, dtype=np.uint8)
        with ThreadPoolExecutor(8) as ex:
            arr = np.stack(list(ex.map(_load, df["path"].tolist())))
        np.save(cf, arr); out.append(arr)
    return out

def load_model(tag):
    r = json.load(open(f"{WORK}/results_{tag}.json"))
    m = timm.create_model(r["arch"], pretrained=False, num_classes=len(classes))
    m.load_state_dict(torch.load(f"{WORK}/ckpt_{tag}.pt", map_location="cpu"))
    return m.to(device).to(memory_format=torch.channels_last).eval()

_mean = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
_std = torch.tensor(STD, device=device).view(1, 3, 1, 1)

@torch.no_grad()
def probs_tta(model, X, bs=32, tta=True):
    """Average softmax over the 8 flip/rotate views (dataset is rotation-invariant)."""
    model.eval(); outs = []
    for i in range(0, len(X), bs):
        x = torch.from_numpy(X[i:i + bs]).to(device).permute(0, 3, 1, 2).float().div_(255)
        x = (x - _mean) / _std
        views = [x]
        if tta:
            views = []
            for flip in (False, True):
                xf = torch.flip(x, dims=(3,)) if flip else x
                for k in range(4):
                    views.append(torch.rot90(xf, k, dims=(2, 3)))
        p = 0
        for v in views:
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model(v.contiguous(memory_format=torch.channels_last))
            p = p + torch.softmax(lg.float(), 1)
        outs.append((p / len(views)).cpu().numpy())
    return np.concatenate(outs)

def acc_f1(y, p):
    pred = p.argmax(1)
    return accuracy_score(y, pred), f1_score(y, pred, average="macro")

def boot_ci(y, pred, n=1000, seed=0):
    rng = np.random.default_rng(seed); N = len(y); fs, accs = [], []
    for _ in range(n):
        i = rng.integers(0, N, N)
        fs.append(f1_score(y[i], pred[i], average="macro")); accs.append((y[i] == pred[i]).mean())
    return np.percentile(fs, [2.5, 97.5]), np.percentile(accs, [2.5, 97.5])

yv = val_df["y"].values; yt = test_df["y"].values
VP = {}   # validation probabilities (with TTA) per model tag
print("helpers ready")


In [ ]:

def train_big(arch, tag, size, bs=16, head_epochs=3, ft_epochs=25, patience=7,
              lr_backbone=1e-4, lr_head=1e-3, wd=0.05, ls=0.1, drop_path=0.1,
              drop_rate=0.2, ema_decay=0.998, seed=42):
    if os.path.exists(f"{WORK}/results_{tag}.json"):
        print(f"[skip] {tag} already trained"); return
    seed_everything(seed)
    Xtr, Xva, _ = cache_for(size)
    tl = GPULoader(Xtr, train_df["y"].values, bs, True, seed=seed)
    vl = GPULoader(Xva, val_df["y"].values, 32, False)
    model = timm.create_model(arch, pretrained=True, num_classes=len(classes),
                              drop_rate=drop_rate, drop_path_rate=drop_path)
    model = model.to(device).to(memory_format=torch.channels_last)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"[{tag}] {arch} | {n_params:.1f}M params | {size}px | bs {bs} | ls {ls} | drop_path {drop_path} | EMA {ema_decay}")

    head_params = list(model.get_classifier().parameters()); hid = {id(p) for p in head_params}
    back_params = [p for p in model.parameters() if id(p) not in hid]
    scaler = torch.amp.GradScaler("cuda"); crit = nn.CrossEntropyLoss(label_smoothing=ls)
    ema = None

    def run_epoch(opt, sched=None):
        model.train(); tot, n, t0 = 0.0, 0, time.time()
        for xb, yb in tl:
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(xb)
            loss = crit(out.float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            if sched: sched.step()
            if ema is not None:
                with torch.no_grad():
                    for pe, pm in zip(ema.parameters(), model.parameters()):
                        pe.lerp_(pm.detach(), 1 - ema_decay)
                    for be, bm in zip(ema.buffers(), model.buffers()):
                        be.copy_(bm)
            tot += loss.item() * len(yb); n += len(yb)
        return tot / n, n / (time.time() - t0)

    # stage 1: head only
    for p in back_params: p.requires_grad = False
    opt = torch.optim.AdamW(head_params, lr=lr_head, weight_decay=wd)
    for ep in range(1, head_epochs + 1):
        trl, ips = run_epoch(opt); v = evaluate(model, vl)
        print(f"ep {ep:02d} [head] loss {trl:.3f} | val acc {v['acc']:.4f} f1 {v['f1']:.4f} | {ips:.0f} img/s")

    # stage 2: full fine-tune, EMA starts here
    for p in back_params: p.requires_grad = True
    ema = copy.deepcopy(model).eval()
    for p in ema.parameters(): p.requires_grad_(False)
    opt = torch.optim.AdamW([{"params": back_params, "lr": lr_backbone}, {"params": head_params, "lr": lr_head}], weight_decay=wd)
    total, warm = ft_epochs * len(tl), len(tl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else max(0.01, 0.5 * (1 + math.cos(math.pi * (s - warm) / (total - warm)))))
    ckpt = f"{WORK}/ckpt_{tag}.pt"; history, best_f1, best_ep, bad = [], -1.0, 0, 0
    for ep in range(head_epochs + 1, head_epochs + ft_epochs + 1):
        trl, ips = run_epoch(opt, sched); v = evaluate(ema, vl)
        history.append({"epoch": ep, "stage": "full", "train_loss": trl, "val_loss": v["loss"], "val_acc": v["acc"], "val_f1": v["f1"]})
        print(f"ep {ep:02d} [full] loss {trl:.3f} | EMA val acc {v['acc']:.4f} f1 {v['f1']:.4f} | {ips:.0f} img/s")
        if v["f1"] > best_f1:
            best_f1, best_ep, bad = v["f1"], ep, 0; torch.save(ema.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= patience:
                print(f"Early stop at epoch {ep} (best {best_ep}, val F1 {best_f1:.4f})"); break

    ema.load_state_dict(torch.load(ckpt, map_location=device))
    p_val = probs_tta(ema, Xva, bs=16, tta=True); a8, f8 = acc_f1(yv, p_val)
    v = evaluate(ema, vl)
    print(f"\nBEST epoch {best_ep} | val acc {v['acc']:.4f} F1 {v['f1']:.4f} (no TTA) | with 8x TTA: acc {a8:.4f} F1 {f8:.4f}")
    VP[tag] = p_val
    res = {"tag": tag, "arch": arch, "params_M": n_params, "best_epoch": best_ep, "val_acc": v["acc"], "val_f1": v["f1"],
           "val_acc_tta": a8, "val_f1_tta": f8, "img_size": size, "label_smoothing": ls, "drop_path": drop_path,
           "ema_decay": ema_decay, "history": history}
    json.dump(res, open(f"{WORK}/results_{tag}.json", "w"))
    np.save(f"{WORK}/valprobs_{tag}.npy", p_val)
    for fn in (f"results_{tag}.json", f"ckpt_{tag}.pt", f"valprobs_{tag}.npy"):
        shutil.copy2(f"{WORK}/{fn}", f"{DST}/{fn}")
    print("Backed up to Drive.")
    del model, ema, tl, vl, opt, sched; gc.collect(); torch.cuda.empty_cache()
print("train_big defined")


In [ ]:
TAG_PREFIX = "repro_r101"

def repro_resnet101(seed, arch="resnet101.tv_in1k", size=224, bs=32, epochs=30, patience=7, lr=1e-4, drop=0.2):
    tag = f"{TAG_PREFIX}_{size}_s{seed}"
    rf = f"{WORK}/results_{tag}.json"
    if os.path.exists(rf):
        print(f"[skip] {tag} already done"); return json.load(open(rf))
    seed_everything(seed)
    Xtr, Xva, Xte = cache_for(size)
    tl = GPULoader(Xtr, train_df["y"].values, bs, True, seed=seed)
    vl = GPULoader(Xva, val_df["y"].values, 64, False)
    model = timm.create_model(arch, pretrained=True, num_classes=len(classes), drop_rate=drop)
    model = model.to(device).to(memory_format=torch.channels_last)
    opt = torch.optim.Adam(model.parameters(), lr=lr)          # Paper 3: Adam, categorical cross-entropy
    crit = nn.CrossEntropyLoss(); scaler = torch.amp.GradScaler("cuda")
    best, best_state, bad, hist = None, None, 0, []
    for ep in range(1, epochs + 1):
        model.train(); tot, n = 0.0, 0
        for xb, yb in tl:
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(xb)
            loss = crit(out.float(), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(yb); n += len(yb)
        v = evaluate(model, vl)
        hist.append({"epoch": ep, "train_loss": tot / n, "val_acc": v["acc"], "val_loss": v["loss"], "val_f1": v["f1"]})
        print(f"[{tag}] ep {ep:02d} train loss {tot/n:.4f} | val acc {v['acc']:.4f} loss {v['loss']:.4f} f1 {v['f1']:.4f}")
        # same selection idea as the paper: best validation accuracy (ties -> lower val loss)
        if best is None or (v["acc"], -v["loss"]) > (best["acc"], -best["loss"]):
            best = {**v, "epoch": ep}; bad = 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: print("early stop"); break
    model.load_state_dict(best_state)
    pt = probs_tta(model, Xte, bs=32, tta=False); ta, tf = acc_f1(yt, pt)     # test, scored once, no TTA
    res = {"tag": tag, "arch": arch, "size": size, "seed": seed, "best_epoch": best["epoch"],
           "val_acc": best["acc"], "val_loss": best["loss"], "val_f1": best["f1"],
           "test_acc": float(ta), "test_f1": float(tf), "history": hist}
    json.dump(res, open(rf, "w")); shutil.copy2(rf, f"{DST}/results_{tag}.json")
    print(f"[{tag}] BEST ep {best['epoch']}: val acc {best['acc']:.4f} loss {best['loss']:.4f} | TEST acc {ta:.4f} F1 {tf:.4f}\n")
    del model, tl, vl, opt; gc.collect(); torch.cuda.empty_cache()
    return res
print("repro function ready")


In [ ]:

import cv2
_K3 = np.ones((3, 3), np.uint8)
def segment_one(rgb, T=127, w=31, C=5):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    g = cv2.GaussianBlur(gray, (5, 5), 0)
    m1 = g > T                                                           # global threshold
    m2 = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, w, C) > 0   # adaptive threshold
    m3 = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] > 0                       # Otsu
    small = cv2.resize(g, (96, 96), interpolation=cv2.INTER_AREA).reshape(-1, 1).astype(np.float32)
    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, _, cen = cv2.kmeans(small, 2, None, crit, 2, cv2.KMEANS_PP_CENTERS)                          # K-means, k=2
    hi, lo = float(cen.max()), float(cen.min())
    gf = g.astype(np.float32); m4 = np.abs(gf - hi) < np.abs(gf - lo)
    votes = m1.astype(np.uint8) + m2.astype(np.uint8) + m3.astype(np.uint8) + m4.astype(np.uint8)
    fused = (votes >= 2).astype(np.uint8)                                                            # majority vote (eq. 1)
    fused = cv2.morphologyEx(fused, cv2.MORPH_OPEN, _K3)                                             # morphological clean-up
    fg = fused * 255
    dist = cv2.distanceTransform(fg, cv2.DIST_L2, 5)                                                 # watershed markers
    sure_fg = (dist > 0.4 * dist.max()).astype(np.uint8) * 255 if dist.max() > 0 else np.zeros_like(fg)
    sure_bg = cv2.dilate(fg, _K3, iterations=2)
    unknown = cv2.subtract(sure_bg, sure_fg)
    _, markers = cv2.connectedComponents(sure_fg)
    markers = (markers + 1).astype(np.int32); markers[unknown == 255] = 0
    markers = cv2.watershed(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), markers)
    final = (fg > 0) & (markers != -1)                                                               # eq. 3
    return (rgb * final[..., None]).astype(np.uint8), float(final.mean())                            # eq. 4

cache_for_orig = cache_for                      # keep the normal (unsegmented) cache loader

def seg_cache_for(size):
    out = []
    for name, X in zip(("train", "val", "test"), cache_for_orig(size)):
        cf = f"{WORK}/cache_{name}_{size}_seg.npy"
        if os.path.exists(cf):
            out.append(np.load(cf)); continue
        with ThreadPoolExecutor(4) as ex:
            res = list(ex.map(segment_one, X))
        arr = np.stack([r[0] for r in res]); cov = np.array([r[1] for r in res])
        print(f"{name}: foreground kept on average {cov.mean()*100:.1f}% | images keeping <5%: {(cov < 0.05).sum()}")
        np.save(cf, arr); out.append(arr)
    return out

# visual check on 8 validation images (saved to Drive too). LOOK AT THIS before training.

import matplotlib.pyplot as plt
Xo = cache_for_orig(384)[1]; Xs = seg_cache_for(384)[1]
idx = np.random.default_rng(0).choice(len(Xo), 8, replace=False)
fig, ax = plt.subplots(2, 8, figsize=(22, 6))
for j, i in enumerate(idx):
    ax[0, j].imshow(Xo[i]); ax[0, j].set_title(classes[val_df["y"].values[i]], fontsize=9); ax[0, j].axis("off")
    ax[1, j].imshow(Xs[i]); ax[1, j].axis("off")
plt.tight_layout(); plt.savefig(f"{DST}/segmentation_examples.png", dpi=110); plt.show()


### Check the picture above first. If the segmented row looks mostly black or destroys the objects, tell me before running the training cells; the paper does not give its threshold parameters, so we may need to tune them.

In [ ]:

# ResNet101 on segmented images (Paper 3 pipeline), 384 px, 3 seeds (about 12 min each)
cache_for = seg_cache_for; TAG_PREFIX = "repro_r101_seg"
Rs = [repro_resnet101(s, size=384, bs=16) for s in (0, 1, 2)]
cache_for = cache_for_orig

va = np.array([r["val_acc"] for r in Rs]); ta = np.array([r["test_acc"] for r in Rs]); tf = np.array([r["test_f1"] for r in Rs])
print("=" * 78)
print(f"ResNet101 + segmentation, 384 px ({len(Rs)} seeds)")
print(f"  best-epoch VAL acc : {va.mean()*100:.2f}% +/- {va.std()*100:.2f}  (max {va.max()*100:.2f}%)")
print(f"  TEST acc / F1      : {ta.mean()*100:.2f}% +/- {ta.std()*100:.2f}  /  {tf.mean():.4f}")
print("Earlier, no segmentation (384 px): VAL 92.65%, TEST 91.16%")
print("Paper 3 ablation   : 92.4% -> 94.7% with segmentation | headline VAL 98.86%")


### Extra: copy Paper 3's training schedule (read off their Figure 8)
Their ResNet101 curves show about 10 epochs with a flat loss (about 0.61), then a sudden drop at epoch 11 and convergence by epoch 20. That looks like 2 phases: backbone frozen for about 10 epochs, then the whole network fine-tuned for about 11 epochs. Our earlier reproduction fine-tuned everything from epoch 1. This cell copies the 2-phase schedule, with and without segmentation. Learning rates are guesses (the paper does not state them).

In [ ]:

def repro_2phase(seed, use_seg, size=384, bs=16, ep1=10, ep2=11, lr1=1e-3, lr2=1e-4, drop=0.2, arch="resnet101.tv_in1k"):
    tag = f"repro2p_r101{'_seg' if use_seg else ''}_{size}_s{seed}"
    rf = f"{WORK}/results_{tag}.json"
    if os.path.exists(rf):
        print(f"[skip] {tag} already done"); return json.load(open(rf))
    seed_everything(seed)
    Xtr, Xva, Xte = (seg_cache_for if use_seg else cache_for_orig)(size)
    tl = GPULoader(Xtr, train_df["y"].values, bs, True, seed=seed)
    vl = GPULoader(Xva, val_df["y"].values, 64, False)
    model = timm.create_model(arch, pretrained=True, num_classes=len(classes), drop_rate=drop)
    model = model.to(device).to(memory_format=torch.channels_last)
    head = list(model.get_classifier().parameters()); hid = {id(q) for q in head}
    back = [q for q in model.parameters() if id(q) not in hid]
    crit = nn.CrossEntropyLoss(); scaler = torch.amp.GradScaler("cuda")
    state = {"best": None, "best_state": None, "hist": []}

    def run_phase(opt, n_epochs, phase, ep0):
        for ep in range(ep0 + 1, ep0 + n_epochs + 1):
            model.train()
            if phase == 1:                       # frozen backbone: keep BatchNorm in inference mode (as Keras does)
                for m in model.modules():
                    if isinstance(m, nn.BatchNorm2d): m.eval()
            tot, n = 0.0, 0
            for xb, yb in tl:
                opt.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=torch.float16):
                    out = model(xb)
                loss = crit(out.float(), yb)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tot += loss.item() * len(yb); n += len(yb)
            v = evaluate(model, vl)
            state["hist"].append({"epoch": ep, "phase": phase, "train_loss": tot / n, "val_acc": v["acc"], "val_loss": v["loss"]})
            print(f"[{tag}] ep {ep:02d} phase {phase} train loss {tot/n:.4f} | val acc {v['acc']:.4f} loss {v['loss']:.4f}")
            b = state["best"]
            if b is None or (v["acc"], -v["loss"]) > (b["acc"], -b["loss"]):
                state["best"] = {**v, "epoch": ep}
                state["best_state"] = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}

    for q in back: q.requires_grad = False
    run_phase(torch.optim.Adam(head, lr=lr1), ep1, 1, 0)                    # phase 1: head only
    for q in back: q.requires_grad = True
    run_phase(torch.optim.Adam(model.parameters(), lr=lr2), ep2, 2, ep1)    # phase 2: everything

    best = state["best"]; model.load_state_dict(state["best_state"])
    pt = probs_tta(model, Xte, bs=32, tta=False); ta, tf = acc_f1(yt, pt)
    res = {"tag": tag, "arch": arch, "size": size, "seed": seed, "best_epoch": best["epoch"], "val_acc": best["acc"],
           "val_loss": best["loss"], "val_f1": best["f1"], "test_acc": float(ta), "test_f1": float(tf), "history": state["hist"]}
    json.dump(res, open(rf, "w")); shutil.copy2(rf, f"{DST}/results_{tag}.json")
    print(f"[{tag}] BEST ep {best['epoch']}: val acc {best['acc']:.4f} loss {best['loss']:.4f} | TEST acc {ta:.4f} F1 {tf:.4f}\n")
    del model, tl, vl; gc.collect(); torch.cuda.empty_cache()
    return res
print("repro_2phase ready")


In [ ]:

# 2-phase ResNet101, 2 seeds each, without and with segmentation (about 20 to 25 min per run)
out = {}
for use_seg in (False, True):
    rs = [repro_2phase(s, use_seg) for s in (0, 1)]
    va = np.array([r["val_acc"] for r in rs]); ta = np.array([r["test_acc"] for r in rs])
    out[use_seg] = (va.mean(), va.max(), ta.mean())
print("=" * 78)
for use_seg, (vm, vx, tm) in out.items():
    print(f"2-phase ResNet101 {'WITH' if use_seg else 'without'} segmentation: VAL mean {vm*100:.2f}% (max {vx*100:.2f}%) | TEST mean {tm*100:.2f}%")
print("Paper 3: VAL 98.86% (Table 3). Table 5 ablation: 92.4% without / 94.7% with segmentation.")


In [ ]:

# Our ConvNeXt-Tiny on segmented images (same recipe as convnext_tiny_384_ls)
base_arch = json.load(open(f"{WORK}/results_convnext_tiny_384_ls.json"))["arch"]
cache_for = seg_cache_for
train_big(base_arch, "cnxtiny_384_seg", 384, bs=16)
cache_for = cache_for_orig


In [ ]:

# Validation-only check: does a segmented-input model help our ensemble?
Xva384 = cache_for_orig(384)[1]
VPv = {}
for t in ["convnext_tiny_384_ls", "cnxbase_384_ls", "cnxsmall_384_ls", "cnxtiny_384_seg"]:
    f = f"{WORK}/valprobs_{t}.npy"
    if os.path.exists(f): VPv[t] = np.load(f)
    else:
        m = load_model(t); VPv[t] = probs_tta(m, Xva384, bs=16); del m; torch.cuda.empty_cache()
print("Validation, 8x TTA:")
for t, p in VPv.items():
    a, f_ = acc_f1(yv, p); print(f"  {t:26s} acc {a:.4f}  F1 {f_:.4f}")
ens3 = ["convnext_tiny_384_ls", "cnxbase_384_ls", "cnxsmall_384_ls"]
for name, members in [("current 3-model ensemble", ens3), ("3-model + segmented Tiny", ens3 + ["cnxtiny_384_seg"])]:
    a, f_ = acc_f1(yv, np.mean([VPv[t] for t in members], axis=0))
    print(f"  {name:26s} acc {a:.4f}  F1 {f_:.4f}   errors {int(round((1 - a) * len(yv)))}/{len(yv)}")
print("Paper 3 headline needs val acc >= 0.9886 (about 8 errors on 712).")
